In [ ]:

import pandas as pd
from pace_pic_validation.io.csv_crunching import units_to_PIC_mmol_m3
from pace_pic_validation.stats.regression import RobustRegression
from pace_pic_validation.io.csv_crunching import units_to_PIC_mmol_m3



In [ ]:
SS20250310_path = '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/LISST_PIC/LISST-PIC 202503 processed.csv'
SS20250602_path =  '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/LISST_PIC/LISST-PIC SS202506 processed.csv'
Chalky_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/LISST_PIC/LISST-PIC-Discrete sample matchup 20260319 1(in)_ICELAND_CHALKY.csv'
Chalky_sensor_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/LISST_PIC/LISST-PIC data 20260319 1(in).csv'

In [ ]:
def linearPICLISST_mmol(Ratiodifference):
    PIC = 14.2781 *Ratiodifference + 0.0447
    return PIC

In [ ]:
chalky_sensor = pd.read_csv(Chalky_sensor_path)



chalky_sensor['PIC_mmolC_m3'] = 179.209 * chalky_sensor['particle'] + 0.330



chalky_sensor = units_to_PIC_mmol_m3(chalky_sensor, start_unit = 'mg/L', start_column = 'PIC_mmolC_m3', end_column ='PIC_mmolC_m3', caco3 = False  )

chalky_sensor['Campaign'] = 'Chalky'
chalky_sensor['research_group'] = 'MarSens, Bigelow'

col_order = ['datetime', 'lat', 'lon', 'research_group', 'Campaign', 'PIC_mmolC_m3']

chalky_sensor = chalky_sensor.drop(
    [col for col in chalky_sensor.columns if col not in col_order],
    axis=1
)

chalky_sensor = chalky_sensor[col_order]

print(chalky_sensor.columns)
chalky_sensor['datetime'] = pd.to_datetime(chalky_sensor['datetime'], format='mixed')
print(chalky_sensor['datetime'].max())


In [ ]:
chalky = pd.read_csv(Chalky_path)
print(chalky.columns)
chalky = chalky.dropna(subset = 'LISST-PIC particle signal')



y_pred, model, intercept, coefficient, r2, rmse, mae = RobustRegression(chalky['LISST-PIC particle signal'], chalky['pic_mean'])


chalky['PIC_mmolC_m3'] = 179.209 * chalky['LISST-PIC particle signal'] + 0.330

print(chalky['PIC_mmolC_m3'].min())
print(chalky['PIC_mmolC_m3'].mean())

chalky = units_to_PIC_mmol_m3(chalky, start_unit = 'mg/L', start_column = 'PIC_mmolC_m3', end_column ='PIC_mmolC_m3', caco3 = False  )

print(chalky['PIC_mmolC_m3'].min())
print(chalky['PIC_mmolC_m3'].mean())

print(chalky.columns)
chalky.drop(columns=['LISST-PIC particle signal', 'MatchingLat', 'MatchingLon', 'MatchingDatetime', 'DateTimeGMT', 'NiskinSubset', 'pic_sd', 'pic_mean'], inplace=True)
chalky.rename(columns={'Station': 'Campaign'}, inplace=True)
chalky['research_group'] = 'MarSens, Bigelow'

col_order = ['datetime', 'lat', 'lon', 'research_group', 'Campaign', 'PIC_mmolC_m3']

chalky = chalky[col_order]
print(chalky.columns)

print


In [ ]:
SS20250310_df = pd.read_csv(SS20250310_path)
SS20250602_df = pd.read_csv(SS20250602_path)

dfs = [SS20250310_df, SS20250602_df]
campaigns = ['SS20250310', 'SS20250602']

for df, campaign in zip(dfs, campaigns): 

    df['research_group'] = 'MarSens'
    df['Campaign'] = campaign


df_total = pd.concat([SS20250310_df, SS20250602_df], ignore_index=True)
df_total.reset_index()

df_total = df_total.rename(columns= {"Datetime": "datetime"})

df_total = df_total.rename(columns= {"Lat": "lat"})

df_total = df_total.rename(columns= {"Lon": "lon"})

df_total = units_to_PIC_mmol_m3(df_total, 'mg/L', 'PIC (microgram C per liter)', 'PIC_mmolC_m3', caco3 = False )
df_total = df_total.drop(columns=['PIC (microgram C per liter)', 'Signet/Refnet particle'])

#df_total.to_csv('/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/LISST_PIC/lisst_pic_all.csv')


ds_list = [df_total, chalky, chalky_sensor]

count = 0
for ds in ds_list:

    print(len(ds))
    count += len(ds)
    print(ds.columns)

print(count)

df_total = pd.concat(ds_list, ignore_index=True)
df_total.reset_index()
print(df_total.columns)

In [ ]:
df_total['datetime'] = pd.to_datetime(df_total['datetime'], format='mixed')
# Define aggregation for each column type
agg_dict = {
    'lat': 'mean',
    'lon': 'mean',
    'PIC_mmolC_m3': 'mean',
    'Campaign': 'first',      # keep first value
    'research_group': 'first',          # keep first ID
}

freq = '5min'

df_binned = df_total.groupby(pd.Grouper(key='datetime', freq=freq)).agg(agg_dict).reset_index()

print(len(df_binned))
df_binned = df_binned.dropna( axis = 0)
print(len(df_binned))

df_binned = df_binned.dropna()
df_binned.to_csv(f'/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/LISST_PIC/lisst_pic_all{freq}.csv')

In [ ]:
print(f'binned in {freq} bins')
print(len(df_binned))

print('original length')
print(len(df_total))

In [ ]:
df_total = pd.concat([df_binned, chalky], ignore_index=True)
df_total = df_total.reset_index(drop=True)
df_total.to_csv(f'/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/LISST_PIC/lisst_pic_all{freq}.csv')
